# Titanic Survival Prediction | 泰坦尼克号生还预测

从 C 语言到机器学习的完整学习记录 —— 一个零基础入门 AI 的自我训练项目

## 核心流程

| Step | 在做什么 | C 语言类比 |
|---|---|---|
| 1. 数据加载 | `pd.read_csv()` 把 CSV 读成带表头的表格 | 类似 `FILE* f = fopen("train.csv", "r")` |
| 2. 缺失值处理 | 用中位数填 NaN，`fillna()` 是向量化操作 | 手写循环给数组里的 -1 赋值 |
| 3. 特征分箱 | `pd.cut` 按数值边界切，`pd.qcut` 按人数均分切 | `if (age<=12) bin=0; else if ...` |
| 4. 特征衍生 | `FamilySize = SibSp + Parch + 1` 算同行人数 | 结构体字段相加 |
| 5. 字符串提取 | `str.extract(r' ([A-Z][a-z]+)\.')` 用正则抠头衔 | 手写字符串解析函数 |
| 6. One-Hot 编码 | `get_dummies()` 把文字 "male"/"female" 转成 0/1 列 | 用 if-else 手动造特征列 |
| 7. 特征对齐 | `reindex(columns=x.columns, fill_value=0)` 保证维度一致 | 保证训练和测试结构体字段完全一致 |
| 8. 训练模型 | `RandomForestClassifier().fit(x, y)` | 调一个 `train(train, answer)` 函数 |
| 9. 预测提交 | `predict()` 生成结果，存成 CSV | `fprintf(f, "...", result)`

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

## 数据导入
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

## 数据处理

### Step 1: 缺失值处理

- **Age（年龄）**：用**全局中位数**填充。中位数对极端值（如 80 岁高龄乘客）不敏感，比均值更真实代表"典型乘客"的年龄。
- **Fare（票价）**：同样用中位数填充。
- **关键细节**：测试集必须用**训练集**算出来的中位数填充，不能直接用测试集自身的 median()，否则会造成**数据泄露（Data Leakage）**——相当于让模型偷看了考试答案。

In [ ]:
## 缺失值处理
# 用训练集的中位数填充（防止数据泄露：测试集不能用自己的统计量）
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = test_data['Age'].fillna(train_data['Age'].median())

train_data['Fare'] = train_data['Fare'].fillna(train_data['Fare'].median())
test_data['Fare'] = test_data['Fare'].fillna(train_data['Fare'].median())

### Step 2: 特征分箱（Binning）

- **Age_Bin（年龄分箱）**：用 `pd.cut` 按**数值边界**等宽切。把连续年龄 0~80 岁切成四段：0-12 儿童、13-18 青少年、19-60 成年、60+ 老年。
  - 为什么分箱？原始年龄有缺失值和离群值，分箱后变成干净的 0/1/2/3 分类，模型更容易学到"儿童优先逃生"这类规则。
- **Fare_Bin（票价分箱）**：用 `pd.qcut` 按**人数比例**等频切（分成 3 档）。
  - 为什么用 qcut 不用 cut？票价分布极度偏斜（大多数人票价便宜，极少数人票价极贵），等宽切会导致某一档挤了 700 多人。等频切保证每档人数差不多，模型才能均衡学习。
  - `r'...'` 原始字符串前缀：`\.` 表示匹配字面意义上的点号，避免被 Python 当转义字符。

In [ ]:
## 分箱
bins = [0, 12, 18, 60, 100]
labels = [0, 1, 2, 3]

# pd.cut: 按数值边界等宽切（年龄）
train_data['Age_Bin'] = pd.cut(train_data['Age'], bins=bins, labels=labels)
test_data['Age_Bin'] = pd.cut(test_data['Age'], bins=bins, labels=labels)

# pd.qcut: 按人数比例等频切（票价，分成3档）
train_data['Fare_Bin'] = pd.qcut(train_data['Fare'], 3, labels=[0, 1, 2])
test_data['Fare_Bin'] = pd.qcut(test_data['Fare'], 3, labels=[0, 1, 2])

### Step 3: 衍生特征 —— 家庭大小 & 独自出行

- **FamilySize = SibSp + Parch + 1**：SibSp（同船兄弟姐妹/配偶数）+ Parch（同船父母/子女数）+ 自己。
- **IsAlone = (FamilySize == 1)**：是否独自出行。

> **为什么有用？** 泰坦尼克号历史规律：小家庭（2~4人）生还率最高——家人互相帮助能快速找到彼此上救生艇；独自出行的乘客没人照应容易被挤散。
>
> **C 语言视角**：`test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1` 相当于给结构体数组新增一个字段，把两个字段的值相加再 +1，一行向量化操作搞定 418 个样本。

In [ ]:
## 家庭大小
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

### Step 4: 从姓名提取头衔（Title）

姓名格式固定为 `"Braund, Owen Harris"`，头衔总是在**逗号后、点号前**。用正则 `r' ([A-Z][a-z]+)\.'` 把头衔抠出来：

| 正则部分 | 含义 |
|---|---|
| ` `（空格） | 匹配逗号后的空格 |
| `[A-Z]` | 匹配一个大写字母（M / D / C）|
| `[a-z]+` | 匹配后续小写字母（r / r / lar）|
| `\.` | 匹配字面点号 |
| `()` 捕获组 | 只返回头衔本身，忽略空格和点号 |

**为什么有用？** 头衔浓缩了社会阶层+性别+年龄信息：Miss/Mrs（女性）生还率高，Master（小男孩）生还率高，Mr（成年男性）生还率低。

**处理复杂头衔**：把稀有头衔（Lady、Countess、Sir、Don、Dona、Mme、Mlle、Ms）统一归并为少数几类，减少噪声。

In [ ]:
## 从姓名提取头衔
# str.extract 是向量化正则提取：对整列每个字符串执行匹配，返回捕获组内容
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Z][a-z]+)\.', expand=False)
test_data['Title'] = test_data['Name'].str.extract(r' ([A-Z][a-z]+)\.', expand=False)

## 处理复杂头衔（归并）
for dataset in [train_data, test_data]:
    dataset['Title'] = dataset['Title'].replace(
        ['Lady', 'Countess', 'Sir', 'Don', 'Dona', 'Mme'], 'Royal')
    dataset['Title'] = dataset['Title'].replace('Mlle', 'Miss')
    dataset['Title'] = dataset['Title'].replace('Ms', 'Miss')

## 特征提取

当前使用的 9 个特征：

| 特征 | 类型 | 来源 | 为什么选它 |
|---|---|---|---|
| Pclass | 分类 | 舱位等级 | 头等舱生还率远高于三等舱 |
| Sex | 分类 | 性别 | 女性生还率远高于男性 |
| SibSp / Parch | 数值 | 亲属数 | 家庭结构影响生还 |
| Age_Bin | 分类 | 年龄分箱 | 儿童/老年优先级不同 |
| Fare_Bin | 分类 | 票价分箱 | 同舱位内高票价位置更靠前 |
| FamilySize | 数值 | 衍生特征 | 小家庭生还率最高 |
| IsAlone | 0/1 | 衍生特征 | 独自出行生还率偏低 |
| Title | 分类 | 姓名提取 | 头衔浓缩阶层+性别+年龄

In [ ]:
## 特征提取
features = ["Pclass", "Sex", "SibSp", "Parch", "Age_Bin", "Fare_Bin", "FamilySize", "Title", "IsAlone"]

## One-Hot 编码 + 维度对齐
# get_dummies: 把文字分类转成 0/1 独立列，模型才能读懂文字
# reindex: 以训练集的列为准，测试集缺的补 0、多的删掉 —— 保证训练和预测的特征维度完全一致
x = pd.get_dummies(train_data[features])
y = train_data["Survived"]  # 0=未生还, 1=生还

x_test = pd.get_dummies(test_data[features])
x_test = x_test.reindex(columns=x.columns, fill_value=0)

## 模型训练与预测

- **RandomForestClassifier(n_estimators=100, random_state=91)**：
  - `n_estimators=100`：森林由 100 棵决策树组成，每棵树投票决定结果，Bagging 降低过拟合。
  - `random_state=91`：**固定随机种子**，保证每次运行的随机分裂过程一致，分数可复现。
  - 为什么之前分数会波动？不固定 random_state 时，决策树的随机采样每次不同，导致准确率上下浮动（0.67~0.77）。

In [ ]:
## 模型设置与训练
# random_state 固定随机种子：保证实验可复现，消除分数波动
model = RandomForestClassifier(n_estimators=100, random_state=91)
model.fit(x, y)

## 预测
predictions = model.predict(x_test)

In [ ]:
## 输出提交文件
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})
submission.to_csv("submission.csv", index=False)
print(f"提交文件已生成，共 {len(submission)} 条预测记录")
print(submission.head())

## 学习收获

✅ 理解了监督学习范式：特征 X → 标签 y → 模型训练 → 预测

✅ 掌握了 pandas 数据预处理全流程：
&nbsp;&nbsp;&nbsp;&nbsp;• 缺失值处理：用**训练集统计量**填充（防泄露）
&nbsp;&nbsp;&nbsp;&nbsp;• 特征分箱：`cut`（等宽）vs `qcut`（等频）的适用场景区别
&nbsp;&nbsp;&nbsp;&nbsp;• 衍生特征：从原始字段中用**向量化运算**构造新信息（FamilySize）
&nbsp;&nbsp;&nbsp;&nbsp;• 字符串特征提取：用**正则表达式**从 Name 中抠出结构化信息（Title）
&nbsp;&nbsp;&nbsp;&nbsp;• One-Hot 编码：`get_dummies` 把文字转 0/1，模型才能读懂
&nbsp;&nbsp;&nbsp;&nbsp;• 特征对齐：`reindex` 保证训练/预测特征维度一致，避免预测报错

✅ 理解了**数据泄露（Data Leakage）**：测试集填充不能用自身统计量，这是一个非常容易踩的坑

✅ 理解了**随机种子 random_state 的作用**：固定伪随机序列，保证实验可复现性

✅ 建立了"从 C 到 Python"的思维迁移：底层数组思维 → 高层数据科学生态的理解

✅ 跑通了 Kaggle 竞赛完整流程

--- 

## 后续优化方向

- [ ] **分组中位数填充**：按 Pclass + Sex 分组填充 Age，比全局中位数更精准
- [ ] **特征交叉**：尝试 Age × Pclass 等组合特征
- [ ] **模型对比**：对比 LogisticRegression / GradientBoosting / XGBoost 的效果
- [ ] **使用 sklearn Pipeline**：把数据预处理和模型训练串成一条管线，规范化流程
- [ ] **特征重要性分析**：用 `model.feature_importances_` 看看模型到底最看重哪些特征